# Run All Diagnostics on Colab GPU

This notebook mounts Google Drive, sets up the project, and runs every diagnostic script against the latest checkpoint.

**Prerequisites:** Upload your project folder and checkpoint to Google Drive.

## 1. Setup: Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === EDIT THIS PATH to match where your project lives in Google Drive ===
PROJECT_DIR = '/content/drive/MyDrive/AutonomousAgents/Project'
CHECKPOINT = f'{PROJECT_DIR}/sac_phase11_lifesteal/checkpoint_ep16200.pt'
# ========================================================================

import os
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Files:', sorted(os.listdir('.'))[:20])

In [ ]:
!pip install -q torch numpy matplotlib pyyaml

In [ ]:
# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), 'GB')

# Verify checkpoint exists
print('\nCheckpoint exists:', os.path.exists(CHECKPOINT))
if os.path.exists(CHECKPOINT):
    print('Size:', round(os.path.getsize(CHECKPOINT) / 1e6, 1), 'MB')

## 2. Environment Correctness Test
Verifies batched env matches single env. Should take ~1 minute.

In [ ]:
!python -m tests.test_batched_correctness

## 3. Spatial Encoding Diagnostic
Tests if Fourier position encoding drives food-seeking behavior. ~5 min.

In [ ]:
!python -m analysis.diagnose_spatial

## 4. Social Discrimination Diagnostic
Tests if the network responds differently to allies vs enemies from ledger history. ~5 min.

In [ ]:
!python -m analysis.diagnose_social --model {CHECKPOINT}

## 5. Friend/Foe Discrimination Test
Tests ally vs enemy classification using exact Phase 9 profiles. ~2 min.

In [ ]:
!python -m analysis.diagnose_friend_foe --model {CHECKPOINT}

## 6. Social Abilities Test Suite
7 behavioral tests with p-values: enemy recognition, ally recognition, cooperation preference, defense, retaliation. ~20 min.

In [ ]:
!python -m analysis.test_social_abilities --model {CHECKPOINT} --quick

## 7. Social Probes (Controlled Ledger Injection)
Injects blank/ally/enemy/forgiveness ledger histories and measures behavioral changes. Saves comparison plots. ~5 min.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # headless backend for plot saving

!python -m analysis.social_probes --checkpoint {CHECKPOINT} --episodes 20 --output-dir probe_results

In [ ]:
# Display the probe result plots
from IPython.display import Image, display
import glob

for png in sorted(glob.glob('probe_results/*.png')):
    print(f'\n--- {png} ---')
    display(Image(filename=png))

## 8. Latest Checkpoint Analysis (10 Greedy Episodes)
Per-agent personality profiles with comparison to earlier training phases. ~5 min.

In [ ]:
!python -m analysis.analyze_latest

## 9. Full Evaluation Across 6 Scenarios
Runs the checkpoint through Free-for-all, Social, 2-Team, 4-Team, Cooperation, and Scarce Resources scenarios. ~10 min.

In [ ]:
# First update eval_config.yaml to point to our checkpoint
eval_config = f"""checkpoint: "{CHECKPOINT}"

profiling:
  episodes: 10

evaluation:
  episodes_per_scenario: 10
  greedy: false

scenarios:
  - name: "Free-for-all"
    type: "social"
    params:
      n_agents: 8
      inject_histories: false
      n_rich_food: 5

  - name: "Social (coherent histories)"
    type: "social"
    params:
      n_agents: 8
      inject_histories: true
      coherent_histories: true
      n_rich_food: 5

  - name: "2-Team Battle"
    type: "team"
    params:
      n_agents: 8
      num_teams: 2

  - name: "4-Team Battle"
    type: "team"
    params:
      n_agents: 8
      num_teams: 4

  - name: "Cooperation Pairs"
    type: "coop"
    params:
      distance: 3
      n_rich_food: 5

  - name: "Scarce Resources"
    type: "social"
    params:
      n_agents: 8
      inject_histories: false
      n_rich_food: 0

output:
  directory: "./eval_results"

device: "auto"

personality:
  aggressive_threshold: 0.15
  cooperative_threshold: 0.20
  forager_threshold: 0.60
  passive_stay_threshold: 0.40
"""

with open('eval_config_colab.yaml', 'w') as f:
    f.write(eval_config)

print('Written eval_config_colab.yaml')

In [ ]:
!python -m analysis.evaluate --config eval_config_colab.yaml

## 10. Replay Analysis (222 Lifesteal Episodes)
Analyzes personality evolution across all saved replays. Only works if replay files are uploaded. ~2 min.

In [ ]:
import glob
replays = glob.glob('results/runs/sac_phase11_lifesteal/replay_ep*.pt')
print(f'Found {len(replays)} replay files')

if len(replays) > 5:
    !python -m analysis.analyze_replays
else:
    print('Not enough replays uploaded — skipping. This data is already in journal/lifesteal_personality_report.md')

## 11. Environment Benchmark
Measures step speed and per-method timing breakdown. ~2 min.

In [ ]:
!python -m tests.benchmark_env

## 12. Action Value & Attention Visualization
Generates per-agent decision visualizations from a saved replay. Needs a replay .pt file.

In [ ]:
import matplotlib
matplotlib.use('Agg')

# Pick a replay file
replay_candidates = glob.glob('results/runs/sac_phase11_lifesteal/replay_ep*.pt') + \
                    glob.glob('results/runs/sac_phase11/replay_ep*.pt') + \
                    glob.glob('replay_ep*.pt')

if replay_candidates:
    replay = replay_candidates[0]
    print(f'Using replay: {replay}')
    !python -m analysis.show_action_values {replay} --frame 64 --output action_values_f64.png
    !python -m analysis.show_action_values {replay} --frame 32 --output action_values_f32.png
    
    from IPython.display import Image, display
    for f in ['action_values_f32.png', 'action_values_f64.png']:
        if os.path.exists(f):
            print(f'\n--- {f} ---')
            display(Image(filename=f))
else:
    print('No replay files found — skipping visualization')

In [ ]:
if replay_candidates:
    !python -m analysis.show_attention_influence {replay} {CHECKPOINT} --frame 64
    
    if os.path.exists('agent_influence.png'):
        from IPython.display import Image, display
        display(Image(filename='agent_influence.png'))
else:
    print('No replay files — skipping attention visualization')

## 13. Collect All Results
Copies everything to a single output folder for easy download.

In [ ]:
import shutil

out = 'all_results'
os.makedirs(out, exist_ok=True)

# Copy all generated images
for pattern in ['*.png', 'probe_results/*.png', 'eval_results/*']:
    for f in glob.glob(pattern):
        dst = os.path.join(out, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f'Copied: {f} -> {dst}')

print(f'\nAll results in: {os.path.abspath(out)}')
print('Files:', sorted(os.listdir(out)))

---
## Done!

**What to copy for the report:**
1. All console outputs above (scroll up and copy text)
2. PNG files from `all_results/` folder
3. `eval_results/results.yaml` if generated

**Estimated total runtime: ~45 minutes on Colab GPU**